# T2B — Medical imaging — hospital / scanner embedding shift

**Lemma D2** · `nuisance="isotropic"` · [Task doc](../../docs/tasks/t02b-chexpert-isotropic.md) · FINAL: `paper_code/T2/Task2B/FINAL.md`

> Chest X-ray E1: best saliency (0.723) and ~9× lower embedding drift vs B0.

| § | What you do |
|---|-------------|
| 1–4 | Install → load demo → `check_applicability` |
| 5–6 | Estimate $\Sigma_{\text{task}}$ → PMH train → Step 5 on deploy holdout |
| 7–8 | Reproduce paper scripts → plug in your data |


**Demo note:** Medical-style σ=0.08 eval noise; full Pneumonia run in paper_code.


## 1 — Install


In [ ]:
!pip install -q matching-pmh torch


## 2 — Config & imports


In [ ]:
import os
import torch
from pmh.benchmark.presets import get_preset
from pmh.pytorch_eval import (
    pytorch_demo_loaders,
    pytorch_isotropic_demo_loaders,
    pytorch_multilayer_vision_demo_loaders,
    pytorch_sequence_demo_loaders,
)
from pmh import PMHConfig, PMHTrainer, evaluate_robust_fit, check_applicability, suggest_nuisance
from pmh.adoption import RECIPE_ONE_LINER, format_recipe_banner

QUICK = os.environ.get("PMH_QUICK", "").lower() in ("1", "true", "yes")
EPOCHS = 2 if QUICK else 6
SEED = 0
print(RECIPE_ONE_LINER)


## 3 — Load demo data


In [ ]:
preset = get_preset("t2b_chexpert_isotropic")
bundle = pytorch_isotropic_demo_loaders(n=N, batch_size=32, seed=SEED, eval_noise_sigma=0.08)
model = bundle.model
hook, head = bundle.encoder, bundle.head
train_loader, src_loader, val_loader = bundle.train_loader, bundle.source_batches, bundle.val_loader
tgt_loader = val_loader  # noisy deploy holdout (sigma=0.08)
print("demo", bundle.n_classes, "classes")


## 4 — Scope (applicability)


In [ ]:
from pmh import check_applicability, suggest_nuisance

print(suggest_nuisance(has_source_labels=True, has_target_domain=False))
app = check_applicability(stack="pytorch", has_target_domain=False)
print(app.summary())


## 5 — Estimate $\Sigma_{\text{task}}$ + PMH train


In [ ]:
import copy
from pmh import PMHTrainer, PMHConfig

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
m = copy.deepcopy(model).to(device)
trainer = PMHTrainer(
    m, hook=hook, head=head, nuisance="isotropic", noise_level=preset.estimate_kwargs["noise_level"], pmh_config=preset.pmh_config, device=device,
)
trainer.fit(train_loader, source_batches=src_loader, epochs=EPOCHS)
print("preflight", trainer.artifact_.preflight, "method", getattr(trainer.artifact_, "method", None))


## 6 — Step 5 (deploy holdout)


In [ ]:
from pmh import evaluate_robust_fit

report = evaluate_robust_fit(
    m, train_loader, val_loader,
    source_batches=src_loader, target_batches=tgt_loader,
    hook=m.enc, head=m.head, nuisance="isotropic", rank=16, 
    pmh_config=preset.pmh_config, epochs=max(2, EPOCHS - 2), include_falsification=False, seed=SEED,
    noise_level=preset.estimate_kwargs["noise_level"],
)
print(report.summary())
if hasattr(report, "baseline_metric"):
    print("deploy holdout — baseline:", report.baseline_metric, "pmh:", report.pmh_metric)


### Embedding drift proxy (paper §4.3)


In [ ]:
import numpy as np
from pmh.features import collect_features
from pmh.tdi import tdi_feature_isotropic
m.eval()
h_clean = collect_features(hook, src_loader, max_batches=10, device=device).cpu().numpy()
parts = [hook(xb.to(device)).detach().cpu().numpy() for xb, _ in val_loader]
h_noisy = np.concatenate(parts, axis=0)
print("tdi proxy clean", tdi_feature_isotropic(h_clean, sigma=0.08))
print("tdi proxy noisy", tdi_feature_isotropic(h_noisy, sigma=0.08))


## 7 — Paper reproduction


Frozen results: `paper_code/T2/Task2B/FINAL.md`

- **Pneumonia chest X-ray — clean test (B0 vs E1 arms):** `python paper_code/T2/Task2B/train.py`
- **Robust eval — Gaussian + acquisition shifts:** `python paper_code/T2/Task2B/eval_robust.py`
- **Saliency stability under noise:** `python paper_code/T2/Task2B/saliency_stability.py`


## 8 — Your pipeline


Swap demo loaders for your `train_loader`, `source_batches`, `target_batches`, and deploy holdout. Hook the backbone before your task head.
